<a href="https://colab.research.google.com/github/pedharangams-beep/-automatic-video-captioning-/blob/main/Welcome_To_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
"""
=============================================================
  FAKE NEWS DETECTION MODEL — AIML Project
  Uses TF-IDF + Passive Aggressive Classifier + Logistic Regression
=============================================================
"""

import numpy as np
import pandas as pd
import re
import string
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import PassiveAggressiveClassifier, LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             classification_report, f1_score)
import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords
import joblib

SAMPLE_DATA = {
    "text": [
        "The president signed a new climate bill today that aims to reduce carbon emissions by 40% over the next decade.",
        "Scientists have discovered a new species of deep-sea fish in the Pacific Ocean.",
        "The stock market closed higher on Friday driven by strong earnings reports from tech companies.",
        "NASA successfully launched its latest Mars rover mission early this morning.",
        "Researchers at Oxford University have published a new study on the effects of sleep deprivation.",
        "The United Nations held an emergency meeting to discuss the ongoing humanitarian crisis.",
        "A 7.2 magnitude earthquake struck off the coast of Japan, prompting tsunami warnings.",
        "The Federal Reserve raised interest rates by 0.25 percent to combat inflation.",
        "New COVID-19 variants are being monitored by health authorities worldwide.",
        "The government announced a $2 billion infrastructure investment plan for rural areas.",
        "Local elections were held in 12 states with record voter turnout reported.",
        "Apple unveiled its latest iPhone lineup at its annual developer conference.",
        "A peace treaty was signed between two neighboring countries after months of negotiations.",
        "Doctors have developed a new surgical technique that reduces recovery time by 50%.",
        "The World Health Organization approved a new vaccine for malaria distribution.",
        "University students protested rising tuition fees in cities across the country.",
        "A wildfire in California has burned over 50,000 acres prompting mass evacuations.",
        "Oil prices dropped sharply after OPEC announced production increases.",
        "The central bank released its quarterly inflation report showing a 3.2% rate.",
        "Tech giant Google reported a 15% increase in quarterly revenue.",
        "BREAKING: Scientists confirm the moon is made of cheese, NASA covers it up!",
        "Drinking bleach cures all viruses and boosts your immune system, doctors hide this!",
        "Bill Gates is microchipping people through COVID vaccines to control the population.",
        "The earth is flat and NASA has been lying to us for decades to steal funding.",
        "SHOCKING: 5G towers are spreading the coronavirus and governments are silent!",
        "Aliens built the pyramids and the Egyptian government is hiding the proof!",
        "Secret cure for cancer suppressed by Big Pharma to protect billion-dollar profits!",
        "Hillary Clinton runs a secret underground pizza shop linked to a child trafficking ring.",
        "The government is poisoning the water supply with fluoride to dumb down citizens.",
        "EXPOSED: The moon landing was filmed in a Hollywood studio by Stanley Kubrick!",
        "Eating raw onions in your socks overnight will cure diabetes and heart disease!",
        "Scientists banned from telling the truth about dinosaurs that still live among us!",
        "URGENT SHARE: 90% of people who took the flu shot died within 2 years — suppressed data!",
        "President secretly replaced by a body double two years ago, insiders reveal!",
        "A lizard-person whistleblower reveals that world leaders are all shape-shifting reptiles.",
        "New study PROVES smartphones cause your brain to melt — media blackout!",
        "Doctors don't want you to know: drinking lemon juice reverses aging completely!",
        "SHOCKING VIDEO: Volcano eruption proves the earth is hollow and inhabited inside!",
        "George Soros is personally funding the migrant crisis to destroy Western civilization!",
        "Top secret document leaked: COVID was engineered in a lab and patents were filed in 1990!",
    ],
    "label": (["REAL"] * 20) + (["FAKE"] * 20)
}

STOP_WORDS = set(stopwords.words("english"))

def preprocess(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    tokens = text.split()
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 2]
    return " ".join(tokens)

def build_dataset(csv_path=None):
    if csv_path:
        df = pd.read_csv(csv_path)
        df = df[["text", "label"]].dropna()
    else:
        df = pd.DataFrame(SAMPLE_DATA)
    df["clean_text"] = df["text"].apply(preprocess)
    return df

def train_models(df):
    X = df["clean_text"]
    y = df["label"]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y)

    tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
    X_train_vec = tfidf.fit_transform(X_train)
    X_test_vec  = tfidf.transform(X_test)

    models = {
        "Passive Aggressive": PassiveAggressiveClassifier(max_iter=1000, random_state=42),
        "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
        "Naive Bayes": MultinomialNB(),
    }

    print("\n" + "="*60)
    print("  MODEL TRAINING & EVALUATION RESULTS")
    print("="*60)

    best_model, best_acc, best_name = None, 0, ""

    for name, clf in models.items():
        clf.fit(X_train_vec, y_train)
        preds = clf.predict(X_test_vec)
        acc = accuracy_score(y_test, preds)
        f1  = f1_score(y_test, preds, pos_label="FAKE")
        print(f"\n📌 {name}")
        print(f"   Accuracy : {acc*100:.2f}%")
        print(f"   F1-Score : {f1:.4f}")
        print(f"   Confusion Matrix:\n{confusion_matrix(y_test, preds)}")
        print(f"   Report:\n{classification_report(y_test, preds)}")
        if acc > best_acc:
            best_acc, best_model, best_name = acc, clf, name

    print("="*60)
    print(f"\n✅ Best Model: {best_name}  (Accuracy: {best_acc*100:.2f}%)")
    return best_model, tfidf, best_name

def predict(news_text, model, vectorizer):
    clean = preprocess(news_text)
    vec   = vectorizer.transform([clean])
    label = model.predict(vec)[0]
    try:
        proba   = model.predict_proba(vec)[0]
        classes = list(model.classes_)
        prob_fake = proba[classes.index("FAKE")] * 100
        prob_real = proba[classes.index("REAL")] * 100
        conf = f"{max(proba)*100:.1f}%"
    except AttributeError:
        prob_fake = prob_real = None
        conf = "N/A"
    return {
        "prediction": label,
        "confidence": conf,
        "prob_FAKE":  f"{prob_fake:.1f}%" if prob_fake else "N/A",
        "prob_REAL":  f"{prob_real:.1f}%" if prob_real else "N/A",
    }

if __name__ == "__main__":
    print("\n🔍 FAKE NEWS DETECTION — AIML PROJECT\n")
    df = build_dataset()
    print(f"Dataset: {len(df)} articles | {df['label'].value_counts().to_dict()}")
    best_model, tfidf, best_name = train_models(df)
    joblib.dump(best_model, "fake_news_model.pkl")
    joblib.dump(tfidf, "tfidf_vectorizer.pkl")

    test_articles = [
        "NASA scientists discover water ice deposits on the Moon's south pole.",
        "EXPOSED: Government hiding secret lizard people in Area 51!",
        "The central bank raised interest rates today amid inflation concerns.",
        "Drinking vinegar makes you live 200 years, doctors hate this trick!",
    ]

    print("\n" + "="*60)
    print("  🧪 LIVE PREDICTION DEMO")
    print("="*60)
    for article in test_articles:
        result = predict(article, best_model, tfidf)
        icon = "✅" if result["prediction"] == "REAL" else "❌"
        print(f"\n{icon} [{result['prediction']}] {article[:70]}...")
        print(f"   Confidence: {result['confidence']} | FAKE: {result['prob_FAKE']} | REAL: {result['prob_REAL']}")







my_news = """
The Great Wall of China is visible from space with the naked eye,
which is actually a common myth, while it is true that the Wall is one of the longest human-made structures ever built.
Bananas grow on trees, another popular belief, is false because banana plants are technically giant herbs.
"""

result = predict(my_news, best_model, tfidf)

print("=" * 50)
print("📰 NEWS TEXT:")
print(my_news.strip())
print("=" * 50)
print(f"🔍 RESULT     : {result['prediction']}")
print(f"📊 Confidence : {result['confidence']}")
print(f"❌ Fake Prob  : {result['prob_FAKE']}")
print(f"✅ Real Prob  : {result['prob_REAL']}")
print("=" * 50)




🔍 FAKE NEWS DETECTION — AIML PROJECT

Dataset: 40 articles | {'REAL': 20, 'FAKE': 20}

  MODEL TRAINING & EVALUATION RESULTS

📌 Passive Aggressive
   Accuracy : 62.50%
   F1-Score : 0.5714
   Confusion Matrix:
[[2 2]
 [1 3]]
   Report:
              precision    recall  f1-score   support

        FAKE       0.67      0.50      0.57         4
        REAL       0.60      0.75      0.67         4

    accuracy                           0.62         8
   macro avg       0.63      0.62      0.62         8
weighted avg       0.63      0.62      0.62         8


📌 Logistic Regression
   Accuracy : 75.00%
   F1-Score : 0.7500
   Confusion Matrix:
[[3 1]
 [1 3]]
   Report:
              precision    recall  f1-score   support

        FAKE       0.75      0.75      0.75         4
        REAL       0.75      0.75      0.75         4

    accuracy                           0.75         8
   macro avg       0.75      0.75      0.75         8
weighted avg       0.75      0.75      0.75         